In [ ]:
import os

os.environ["ROBOFLOW_API_KEY"] = ""


In [ ]:
import gc
import inspect
import sys
import weakref
from pathlib import Path

import matplotlib.pyplot as plt
import supervision as sv
import torch
import torchvision.transforms as T
from PIL import Image
from rfdetr import RFDETRBase
from rfdetr.util.misc import NestedTensor
from supervision.metrics import MeanAveragePrecision
from tqdm import tqdm

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("path/to/basketball-player-detection-2.v13i.coco")
OUTPUT_DIR   = Path("path/to/output/basketball_cka")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS           = 50
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 16
LEARNING_RATE    = 1e-4

# ── CKA regularization ────────────────────────────────────────────────────────
# Set to 0.0 to disable CKA and reproduce the baseline exactly.
# Recommended range: 0.1 – 1.0. 
CKA_LAMBDA = 0.5

# ── Inference ─────────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.5

print("=" * 50)
print("RF-DETR + CKA — BASKETBALL CONFIGURATION")
print("=" * 50)
print(f"  Dataset root : {DATASET_ROOT}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LEARNING_RATE}")
print(f"  CKA lambda   : {CKA_LAMBDA}  {'(enabled)' if CKA_LAMBDA > 0 else '(disabled — baseline mode)'}")

In [ ]:
for mod in ["rfdetr.models.lwdetr", "rfdetr.models"]:
    if mod in sys.modules:
        del sys.modules[mod]

import rfdetr.models.lwdetr

print(f"lwdetr.py : {rfdetr.models.lwdetr.__file__}")

source = inspect.getsource(rfdetr.models.lwdetr.LWDETR.forward)
if "backbone_features" in source:
    print("✓ backbone_features found in LWDETR.forward()")
else:
    print("✗ backbone_features NOT found — please install the modified lwdetr.py")

# ── Forward-pass shape check ───────────────────────────────────────────────────
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_dbg_model = RFDETRBase().model.model.to(device)

_test_img_path = next(
    f for f in (DATASET_ROOT / "train").iterdir()
    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
_img    = Image.open(_test_img_path).convert("RGB").resize((448, 448))
_tensor = T.ToTensor()(_img).unsqueeze(0).to(device)
_mask   = torch.zeros((1, 448, 448), dtype=torch.bool, device=device)

with torch.no_grad():
    _out = _dbg_model(NestedTensor(_tensor, _mask))

if "backbone_features" in _out:
    print(f"\n✓ backbone_features confirmed (device: {device}):")
    for k, v in _out["backbone_features"].items():
        print(f"  Scale {k}: {list(v.shape)}")
else:
    print("\n✗ backbone_features absent from output dict.")

del _dbg_model, _tensor, _mask, _out

In [ ]:
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem   = props.total_memory / 1024 ** 3
        print(f"  GPU {i} : {props.name}  ({mem:.1f} GB)")

In [ ]:
model   = RFDETRBase()
history = []

def _on_epoch_end(data: dict):
    history.append(data)
    map_val = (
        data.get("test_coco_eval_bbox", [0])[0]
        if "test_coco_eval_bbox" in data
        else data.get("test_map", 0)
    )
    msg = (
        f"  Epoch {data.get('epoch', 0):>3}  |  "
        f"Train: {data.get('train_loss', 0):.4f}  |  "
        f"Val: {data.get('test_loss', 0):.4f}  |  "
        f"mAP: {map_val:.4f}"
    )
    if "train_loss_cka" in data:
        msg += f"  |  CKA: {data['train_loss_cka']:.4f}"
    print(msg)

model.callbacks["on_fit_epoch_end"].append(_on_epoch_end)
print("Model initialised — starting training ...\n")
print(f"  CKA lambda = {CKA_LAMBDA}\n")

model.train(
    dataset_dir             = str(DATASET_ROOT),
    epochs                  = EPOCHS,
    batch_size              = BATCH_SIZE,
    grad_accum_steps        = GRAD_ACCUM_STEPS,
    lr                      = LEARNING_RATE,
    output_dir              = str(OUTPUT_DIR),
    use_ema                 = True,
    tensorboard             = False,
    early_stopping          = True,
    early_stopping_patience = 10,
    amp                     = True,
    device                  = "cuda",
    num_workers             = 0,
    multi_scale             = False,
    resolution              = 448,
    cka_lambda              = CKA_LAMBDA,
)

print(f"\n✅ Training complete  |  Checkpoints → {OUTPUT_DIR}")

In [ ]:
metrics_img_path = OUTPUT_DIR / "metrics_plot.png"

if metrics_img_path.exists():
    plt.figure(figsize=(14, 6))
    plt.imshow(Image.open(metrics_img_path))
    plt.axis("off")
    plt.title("Training Metrics (RF-DETR metrics_plot.png)", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠ metrics_plot.png not found at {metrics_img_path}")

if history:
    import pandas as pd
    df = pd.DataFrame(history)

    has_cka = "train_loss_cka" in df.columns
    n_cols  = 3 if has_cka else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 4))

    axes[0].plot(df["epoch"], df["train_loss"], label="Train",      marker="o", linewidth=2)
    axes[0].plot(df["epoch"], df["test_loss"],  label="Validation", marker="o", linewidth=2, linestyle="--")
    axes[0].set_title("Train / Validation Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    map_col  = "test_coco_eval_bbox" if "test_coco_eval_bbox" in df.columns else "test_map"
    if map_col in df.columns:
        map_vals = df[map_col].apply(lambda x: x[0] if isinstance(x, list) else x)
        axes[1].plot(df["epoch"], map_vals, color="#27ae60",
                     label="mAP@0.5:0.95", marker="o", linewidth=2)
        axes[1].set_title("Validation mAP", fontweight="bold")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

    if has_cka:
        axes[2].plot(df["epoch"], df["train_loss_cka"], color="#e74c3c",
                     label="CKA loss", marker="o", linewidth=2)
        axes[2].set_title("CKA Regularization Loss", fontweight="bold")
        axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("CKA Loss")
        axes[2].legend(); axes[2].grid(True, alpha=0.3)

    fig.suptitle(
        f"RF-DETR + CKA — Basketball  (λ={CKA_LAMBDA})",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    save_path = OUTPUT_DIR / "training_curves.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved → {save_path}")

In [ ]:
def cleanup_gpu(obj=None, verbose: bool = True):
    if not torch.cuda.is_available():
        print("No GPU detected — skipping cleanup.")
        return

    torch.cuda.synchronize()
    if verbose:
        alloc = torch.cuda.memory_allocated() / 1024 ** 2
        resv  = torch.cuda.memory_reserved()  / 1024 ** 2
        print(f"Before  allocated: {alloc:.1f} MB  |  reserved: {resv:.1f} MB")

    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("⚠ Object may still have references elsewhere.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()

    if verbose:
        alloc = torch.cuda.memory_allocated() / 1024 ** 2
        resv  = torch.cuda.memory_reserved()  / 1024 ** 2
        print(f"After   allocated: {alloc:.1f} MB  |  reserved: {resv:.1f} MB")


cleanup_gpu(model, verbose=True)

In [ ]:
CHECKPOINT = OUTPUT_DIR / "checkpoint_best_total.pth"

model = RFDETRBase(pretrain_weights=str(CHECKPOINT))
model.optimize_for_inference()
print(f"✓ Model loaded from: {CHECKPOINT}")

ds = sv.DetectionDataset.from_coco(
    images_directory_path = str(DATASET_ROOT / "test"),
    annotations_path      = str(DATASET_ROOT / "test" / "_annotations.coco.json"),
)
print(f"✓ Test set: {len(ds)} images  |  Classes: {ds.classes}")

In [ ]:
targets, predictions = [], []

for path, _, annotations in tqdm(ds, desc="Evaluating"):
    image      = Image.open(path).convert("RGB")
    detections = model.predict(image, threshold=0)
    targets.append(annotations)
    predictions.append(detections)

metric     = MeanAveragePrecision()
map_result = metric.update(predictions, targets).compute()

print("\n" + "=" * 45)
print(f"TEST RESULTS — Basketball CKA  (λ={CKA_LAMBDA})")
print("=" * 45)
print(f"  mAP@0.5:0.95 : {map_result.map50_95:.4f}")
print(f"  mAP@0.5      : {map_result.map50:.4f}")
print(f"  mAP@0.75     : {map_result.map75:.4f}")

In [ ]:
path, _, annotations = ds[0]
image      = Image.open(path).convert("RGB")
detections = model.predict(image, threshold=CONFIDENCE_THRESHOLD)

print(f"Image : {Path(path).name}")
print(f"GT    : {len(annotations)} objects")
print(f"Pred  : {len(detections)} detections  (threshold={CONFIDENCE_THRESHOLD})")

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness  = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
palette    = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00",
])

bbox_ann  = sv.BoxAnnotator(color=palette, thickness=thickness)
label_ann = sv.LabelAnnotator(color=palette, text_color=sv.Color.BLACK,
                               text_scale=text_scale)

gt_labels   = [ds.classes[c] for c in annotations.class_id]
pred_labels = [f"{ds.classes[c]} {conf:.2f}"
               for c, conf in zip(detections.class_id, detections.confidence)]

gt_img   = label_ann.annotate(bbox_ann.annotate(image.copy(), annotations), annotations, gt_labels)
pred_img = label_ann.annotate(bbox_ann.annotate(image.copy(), detections),  detections,  pred_labels)

sv.plot_images_grid(
    images    = [gt_img, pred_img],
    grid_size = (1, 2),
    titles    = ["Ground Truth", f"RF-DETR + CKA (conf ≥ {CONFIDENCE_THRESHOLD})"],
)